# 04 — Data Preparation: Adding 2024/2025 Legislative Results
Extends the main feature table with 2024 and 2025 legislative election results by municipality.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
import shap
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('tabela_principal_autarquicas_2025_v0.csv', sep=';')

In [ ]:
print(df.columns.tolist())

In [ ]:
# acrescentar "_aut_2021" a todas as colunas que começam por "Partido_Vencedor"
df.rename(
    columns=lambda c: f"{c}_aut_2021" if c.startswith("Partido_Vencedor_") else c,
    inplace=True
)


In [ ]:
df_legis_2024 = pd.read_csv('resultados_legislativas_2024.csv', sep=',')
df_legis_2025 = pd.read_csv('resultados_legislativas_2025.csv', sep=',')

In [ ]:
# 1️⃣ escolher as colunas com resultados dos partidos
partidos_cols = df_legis_2024.columns[1:]   # começa na 2ª coluna (0,1...)

# 2️⃣ tentar converter para número
convertido = df_legis_2024[partidos_cols].apply(pd.to_numeric, errors="coerce")

# 3️⃣ marcar onde o pandas não conseguiu converter
mascara = convertido.isna() & df_legis_2024[partidos_cols].notna()

# 4️⃣ percorrer e imprimir valores inválidos
for col in partidos_cols:
    invalidos = df_legis_2024.loc[mascara[col], col].unique()
    if len(invalidos) > 0:
        print(f"{col}: {invalidos}")


In [ ]:
# Use only the party columns for max calculations
df_legis_2024["Partido_Vencedor_2024"] = df_legis_2024[partidos_cols].idxmax(axis=1)
df_legis_2024["Percentagem_Max_2024"] = df_legis_2024[partidos_cols].max(axis=1)


In [ ]:
# 1️⃣ escolher as colunas com resultados dos partidos
partidos_cols = df_legis_2025.columns[1:]   # começa na 2ª coluna (0,1...)

# 2️⃣ tentar converter para número
convertido = df_legis_2025[partidos_cols].apply(pd.to_numeric, errors="coerce")

# 3️⃣ marcar onde o pandas não conseguiu converter
mascara = convertido.isna() & df_legis_2025[partidos_cols].notna()

# 4️⃣ percorrer e imprimir valores inválidos
for col in partidos_cols:
    invalidos = df_legis_2025.loc[mascara[col], col].unique()
    if len(invalidos) > 0:
        print(f"{col}: {invalidos}")


In [ ]:
# Use only the party columns for max calculations
df_legis_2025["Partido_Vencedor_2025"] = df_legis_2025[partidos_cols].idxmax(axis=1)
df_legis_2025["Percentagem_Max_2025"] = df_legis_2025[partidos_cols].max(axis=1)

### Partir colunas a meio: resultados + partido vencedor

In [ ]:
resultados_2024 = df_legis_2024.drop(columns=["Partido_Vencedor_2024", "Percentagem_Max_2024"])
resultados_2024.rename(
    columns={c: f"{c}_2024" for c in resultados_2024.columns[1:]},
    inplace=True
)


In [ ]:
resultados_2025 = df_legis_2025.drop(columns=["Partido_Vencedor_2025", "Percentagem_Max_2025"])
resultados_2025.rename(
    columns={c: f"{c}_2025" for c in resultados_2025.columns[1:]},
    inplace=True
)

In [ ]:
partido_vencedor_2024 = df_legis_2024[["Concelho", "Partido_Vencedor_2024"]]
partido_vencedor_2025 = df_legis_2025[["Concelho", "Partido_Vencedor_2025"]]

In [ ]:
partido_vencedor_2024 = pd.get_dummies(partido_vencedor_2024, columns=["Partido_Vencedor_2024"], prefix="PV_2024", dtype=int)
partido_vencedor_2025 = pd.get_dummies(partido_vencedor_2025, columns=["Partido_Vencedor_2025"], prefix="PV_2025", dtype=int)


#### Juntar à tabela principal anterior

In [ ]:
from functools import reduce

nova_tabela_principal = reduce(
	lambda left, right: pd.merge(left, right, on='Concelho', how='outer'),
	[partido_vencedor_2024, partido_vencedor_2025, resultados_2024, resultados_2025, df]
)

##### Guardar tabelas

In [ ]:
nova_tabela_principal.to_csv('tabela_principal_2023_2025_legislativas_v0.csv', sep=';', index=False)
resultados_2024.to_csv('resultados_legislativas_2024_v0.csv', sep=';', index=False)
resultados_2025.to_csv('resultados_legislativas_2025_v0.csv', sep=';', index=False)
